# Notebook 6: Master Evaluation
## Project: Multi-Paradigm AI vs. Human Text Detection

### Objective
To conduct a rigorous, head-to-head evaluation of all trained paradigms (ML, DL, SOTA) on a unified test set. 

### Beyond Accuracy: Cost & Sustainability
In modern MLOps, accuracy is only one dimension of model viability. We evaluate our models on two critical axes:
1. **F1-Score:** To measure the balance of Precision and Recall.
2. **Inference Latency (ms):** The real-world time cost for a user waiting for a prediction.
3. **Size (MB):** The disk space occupied to load the models

In [56]:
# Imporitng Libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import time
import os
import re
from sklearn.metrics import accuracy_score,precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer,BertForSequenceClassification

In [57]:
# Loading data and sampling for test
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
df=pd.read_csv("/kaggle/input/datasets/matejkore/ai-detector-dataset/ai_detector_dataset.csv")
df.head()

,text,label
0,"Got take-out. Very friendly staff, reasonable ...",Human
1,Love this bar! Fun crowd and the staff are all...,Human
2,"After watching Whale Wars: Viking Shores, I ca...",Human
3,Kelly really wanted the new iPhone. She begged...,Human
4,Though this is the easiest way to secure a spo...,Human


In [58]:
# Replacing author names with binary labels
df['label']=[1 if author=='AI' else 0 for author in df['label']]
df.head()

,text,label
0,"Got take-out. Very friendly staff, reasonable ...",0
1,Love this bar! Fun crowd and the staff are all...,0
2,"After watching Whale Wars: Viking Shores, I ca...",0
3,Kelly really wanted the new iPhone. She begged...,0
4,Though this is the easiest way to secure a spo...,0


In [59]:
# Cleaning the texts
def text_cleaning(text):
    text=text.lower()
    text=re.sub(r'\n',' ',text)
    text=re.sub(r'\s+',' ',text)
    return text
df['text']=df['text'].apply(text_cleaning)
df.head()

,text,label
0,"got take-out. very friendly staff, reasonable ...",0
1,love this bar! fun crowd and the staff are all...,0
2,"after watching whale wars: viking shores, i ca...",0
3,kelly really wanted the new iphone. she begged...,0
4,though this is the easiest way to secure a spo...,0


In [60]:
# Dropping duplicates
df=df.drop_duplicates(subset='text')
df.duplicated(subset='text').sum()

np.int64(0)

In [61]:
# Splitting the dataset into train and text split
X_train,X_test,y_train,y_test=train_test_split(df['text'],df['label'],test_size=0.2,random_state=42,stratify=df['label'])
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((265651,), (265651,), (66413,), (66413,))

In [62]:
test_df=pd.DataFrame()
_,test_df['text'],_,test_df['label']=train_test_split(X_test,y_test,test_size=0.99,random_state=42,stratify=y_test)

In [63]:
# Loading Logistic Regression and TFIDF
lr_model=joblib.load("/kaggle/input/datasets/antareepghosh18/ai-detector-dataset/logistic_regression_model.joblib")
tfidf=joblib.load("/kaggle/input/datasets/antareepghosh18/ai-detector-dataset/tfidf_vectorizer.joblib")

In [64]:
# Loading LSTM and Vocab
class LSTMModel(nn.Module):
    def __init__(self,vocab_size,embed_dim=128,hidden_dim=128):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embed_dim)
        self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True)
        self.linear=nn.Linear(hidden_dim,1)
    def forward(self,x):
        x=self.embedding(x)
        _,(hidden,_)=self.lstm(x)
        out=self.linear(hidden[-1])
        return torch.sigmoid(out)
vocab=torch.load("/kaggle/input/datasets/antareepghosh18/ai-detector-dataset/vocab.pth")
lstm_model=LSTMModel(len(vocab),128,128)
lstm_model.load_state_dict(torch.load("/kaggle/input/datasets/antareepghosh18/ai-detector-dataset/lstm_model.pth",map_location=device))
lstm_model.eval().to(device)

LSTMModel(
  (embedding): Embedding(20000, 128)
  (lstm): LSTM(128, 128, batch_first=True)
  (linear): Linear(in_features=128, out_features=1, bias=True)
)

In [65]:
# Loading BERT
bert_tokenizer=BertTokenizer.from_pretrained("/kaggle/input/datasets/antareepghosh18/ai-detector-dataset")
bert_model=BertForSequenceClassification.from_pretrained("/kaggle/input/datasets/antareepghosh18/ai-detector-dataset").to(device)
bert_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [66]:
# Defining predcition pipeline
def preprocess_pytorch_text(text, max_len=350):
    cleaned_text=text_cleaning(text)
    tokens=[vocab.get(word, vocab["<UNK>"]) for word in cleaned_text.split()]
    if len(tokens)<max_len:
        tokens=tokens+[0]*(max_len-len(tokens))
    else:
        tokens=tokens[:max_len]
    return torch.tensor([tokens]).to(device)
def bert_predict_batch(texts,batch_size=16):
    all_preds=[]
    for i in range(0,len(texts),batch_size):
        batch_texts=texts[i:i+batch_size]
        inputs=bert_tokenizer(batch_texts, 
                              padding=True, 
                              truncation=True, 
                              max_length=200, 
                              return_tensors="pt").to(device)
        with torch.no_grad():
            outputs=bert_model(**inputs)
            batch_preds=torch.argmax(outputs.logits,dim=1).cpu().numpy()
            all_preds.extend(batch_preds)
        torch.cuda.empty_cache() 
    return np.array(all_preds)

In [67]:
# Defining the evaluation loop
results={}
def evaluate_paradigm(name,y_true,y_pred,model_path,predict_fn_single,sample_text):
    acc=accuracy_score(y_true,y_pred)
    prec,rec,f1,_=precision_recall_fscore_support(y_true,y_pred,average='binary')
    if os.path.isdir(model_path):
        size_mb=sum(os.path.getsize(os.path.join(model_path,f)) for f in os.listdir(model_path))/(1024*1024)
    else:
        size_mb=os.path.getsize(model_path)/(1024*1024)      
    start_time=time.time()
    for _ in range(50):
        _=predict_fn_single(sample_text)
    latency=((time.time()-start_time)/50)*1000
    return [acc,f1,prec,rec,latency,size_mb]

In [68]:
# Running Evaluations
sample="This is a sample text to measure latency."
print("Running LR...")
lr_preds=lr_model.predict(tfidf.transform(test_df['text'].str.lower()))
results['Logistic Regression']=evaluate_paradigm("LR",test_df['label'],lr_preds, 
                                                 "/kaggle/input/datasets/antareepghosh18/ai-detector-dataset/logistic_regression_model.joblib", 
                                                lambda x: lr_model.predict(tfidf.transform([x])),sample)
print("Running LSTM...")
lstm_preds=[1 if lstm_model(preprocess_pytorch_text(t)).item()>0.5 else 0 for t in test_df['text']]
results['LSTM']=evaluate_paradigm("LSTM", test_df['label'],lstm_preds, 
                                             "/kaggle/input/datasets/antareepghosh18/ai-detector-dataset/lstm_model.pth", 
                                             lambda x: lstm_model(preprocess_pytorch_text(x)), sample)

print("Running BERT...")
bert_preds=bert_predict_batch(test_df['text'].tolist())
results['BERT']=evaluate_paradigm("BERT",test_df['label'],bert_preds, 
                                          "/kaggle/input/datasets/antareepghosh18/ai-detector-dataset",
                                          lambda x: bert_predict_batch([x]), sample)

Running LR...
Running LSTM...
Running BERT...


In [69]:
# Creating performance table
performance_df=pd.DataFrame(results, index=['Accuracy', 'F1-Score', 'Precision', 'Recall', 'Latency (ms)', 'Size (MB)']).T
performance_df

,Accuracy,F1-Score,Precision,Recall,Latency (ms),Size (MB)
Logistic Regression,0.825594,0.825833,0.825018,0.826649,0.568929,0.153411
LSTM,0.901291,0.903287,0.885707,0.921580,1.118307,10.272761
BERT,0.944516,0.944702,0.941904,0.947517,8.049855,440.039149


### Final Architecture Conclusions
* **The Speed/Accuracy Trade-off:** Logistic Regression operates in ~0.5ms with a negligible memory footprint, making it ideal for high-volume, low-stakes filtering.
* **The Transformer Cost:** While BERT achieved the highest F1-Score, its inference latency is significantly higher. 
* **Production Recommendation:** For a live application, a **hybrid cascade architecture** is recommended: using Logistic Regression for rapid initial screening, and escalating to BERT only when the baseline confidence is low.